# 🏠 House Price Prediction using Machine Learning

End-to-end regression project covering EDA, preprocessing, Linear/Ridge/Lasso Regression, cross-validation, hyperparameter optimization, ablation study, and feature engineering.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [ ]:
from pathlib import Path
path = Path('Housing.csv')
if not path.exists():
    raise FileNotFoundError('Run download_dataset.py first or place Housing.csv beside this notebook.')
df = pd.read_csv(path)
df.head()

## 3. Understand the Dataset

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.info()
display(df.describe())
print('Missing values:
', df.isnull().sum())
print('Duplicates:', df.duplicated().sum())

## 4. Exploratory Data Analysis (EDA)
Univariate, bivariate and multivariate analysis are used to understand distributions, relationships and correlations.

In [ ]:
plt.figure(figsize=(8,5)); sns.histplot(df['price'], kde=True); plt.title('Distribution of House Prices'); plt.show()
plt.figure(figsize=(8,5)); sns.boxplot(x=df['price']); plt.title('Boxplot of House Prices'); plt.show()
plt.figure(figsize=(8,5)); sns.scatterplot(data=df, x='area', y='price'); plt.title('House Price vs Area'); plt.show()
plt.figure(figsize=(8,5)); sns.boxplot(data=df, x='bedrooms', y='price'); plt.title('House Price vs Bedrooms'); plt.show()
plt.figure(figsize=(9,5)); sns.boxplot(data=df, x='furnishingstatus', y='price'); plt.title('House Price vs Furnishing Status'); plt.show()
plt.figure(figsize=(10,7)); sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, cmap='coolwarm'); plt.title('Correlation Heatmap'); plt.show()

## 5. Train-Test Split and Preprocessing

In [ ]:
X = df.drop('price', axis=1)
y = df['price']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
numeric_features = ['area','bedrooms','bathrooms','stories','parking']
categorical_features = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea','furnishingstatus']
preprocessor = ColumnTransformer([('num', StandardScaler(), numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

## 6. Linear Regression Baseline

In [ ]:
from sklearn.linear_model import LinearRegression
linear_pipeline = Pipeline([('preprocessing', preprocessor), ('model', LinearRegression())])
linear_pipeline.fit(X_train, y_train)
y_pred = linear_pipeline.predict(X_test)

## 7. Model Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f'MAE: {mae:,.2f}')
print(f'MSE: {mse:,.2f}')
print(f'RMSE: {rmse:,.2f}')
print(f'R²: {r2:.4f}')

In [ ]:
plt.figure(figsize=(8,6)); sns.scatterplot(x=y_test, y=y_pred); plt.plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],linestyle='--'); plt.xlabel('Actual Price'); plt.ylabel('Predicted Price'); plt.title('Actual vs Predicted House Prices'); plt.show()
residuals = y_test - y_pred
plt.figure(figsize=(8,5)); sns.scatterplot(x=y_pred, y=residuals); plt.axhline(0, linestyle='--'); plt.xlabel('Predicted Price'); plt.ylabel('Residual'); plt.title('Residuals vs Predicted Prices'); plt.show()

## 8. 5-Fold Cross Validation

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(linear_pipeline, X_train, y_train, cv=kfold, scoring='r2')
print('Fold R²:', cv_scores)
print(f'Mean CV R²: {cv_scores.mean():.4f}')
print(f'Std: {cv_scores.std():.4f}')

## 9. Ridge Regression + Hyperparameter Optimization

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
ridge_pipeline = Pipeline([('preprocessing', preprocessor), ('model', Ridge())])
ridge_search = GridSearchCV(ridge_pipeline, {'model__alpha':[0.01,0.1,1,10,100]}, cv=5, scoring='r2', n_jobs=-1)
ridge_search.fit(X_train, y_train)
ridge_pred = ridge_search.predict(X_test)
ridge_mae = mean_absolute_error(y_test,ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test,ridge_pred))
ridge_r2 = r2_score(y_test,ridge_pred)
print('Best alpha:', ridge_search.best_params_)
print(f'Best CV R²: {ridge_search.best_score_:.4f}')
print(f'Ridge R²: {ridge_r2:.4f}')

## 10. Lasso Regression + Hyperparameter Optimization

In [ ]:
from sklearn.linear_model import Lasso
lasso_pipeline = Pipeline([('preprocessing', preprocessor), ('model', Lasso(max_iter=100000))])
lasso_search = GridSearchCV(lasso_pipeline, {'model__alpha':[0.01,0.1,1,10,100]}, cv=5, scoring='r2', n_jobs=-1)
lasso_search.fit(X_train,y_train)
lasso_pred = lasso_search.predict(X_test)
lasso_mae = mean_absolute_error(y_test,lasso_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_test,lasso_pred))
lasso_r2 = r2_score(y_test,lasso_pred)
print('Best alpha:', lasso_search.best_params_)
print(f'Best CV R²: {lasso_search.best_score_:.4f}')
print(f'Lasso R²: {lasso_r2:.4f}')

## 11. Ablation Study: Remove Area

In [ ]:
X_no_area = X.drop('area', axis=1)
Xtr, Xte, ytr, yte = train_test_split(X_no_area, y, test_size=0.2, random_state=42)
prep_no_area = ColumnTransformer([('num', StandardScaler(), ['bedrooms','bathrooms','stories','parking']), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])
no_area_model = Pipeline([('preprocessing', prep_no_area), ('model', LinearRegression())])
no_area_model.fit(Xtr,ytr)
no_area_pred = no_area_model.predict(Xte)
r2_no_area = r2_score(yte,no_area_pred)
print(f'R² with area: {r2:.4f}')
print(f'R² without area: {r2_no_area:.4f}')

## 12. Feature Engineering Experiment

In [ ]:
X_imp = X.copy()
X_imp['total_rooms'] = X_imp['bedrooms'] + X_imp['bathrooms']
X_imp['area_per_bedroom'] = X_imp['area'] / X_imp['bedrooms']
Xtr, Xte, ytr, yte = train_test_split(X_imp,y,test_size=0.2,random_state=42)
num_imp = numeric_features + ['total_rooms','area_per_bedroom']
prep_imp = ColumnTransformer([('num',StandardScaler(),num_imp),('cat',OneHotEncoder(handle_unknown='ignore'),categorical_features)])
imp_model = Pipeline([('preprocessing',prep_imp),('model',LinearRegression())])
imp_model.fit(Xtr,ytr)
imp_pred = imp_model.predict(Xte)
improved_mae = mean_absolute_error(yte,imp_pred)
improved_rmse = np.sqrt(mean_squared_error(yte,imp_pred))
improved_r2 = r2_score(yte,imp_pred)
print(f'Improved MAE: {improved_mae:,.2f}')
print(f'Improved RMSE: {improved_rmse:,.2f}')
print(f'Improved R²: {improved_r2:.4f}')

## 13. Final Model Comparison

In [ ]:
comparison = pd.DataFrame({'Model':['Linear Regression','Ridge Regression','Lasso Regression','Feature-Engineered Linear Regression'],'MAE':[mae,ridge_mae,lasso_mae,improved_mae],'RMSE':[rmse,ridge_rmse,lasso_rmse,improved_rmse],'R²':[r2,ridge_r2,lasso_r2,improved_r2]})
display(comparison.sort_values('R²',ascending=False).reset_index(drop=True))
best = comparison.loc[comparison['R²'].idxmax()]
print('Best Model:', best['Model'])
print(f"Best R²: {best['R²']:.4f}")

## 14. Conclusion

In [ ]:
print('The project demonstrates an end-to-end regression workflow with preprocessing, regularization, cross-validation, hyperparameter tuning, ablation analysis and feature engineering.')